# Building a QA Agent

In this exercise, you will build a basic Question Answering (QA) agent using OpenAI's gpt-4o-mini model. You will start by directly interacting with the model to analyze a PDF document containing health insurance policy details. Then, you will refactor this logic into a reusable Python class, laying the groundwork for the next exercise where you will wrap this agent in an Agent2Agent (A2A) server.

## Import Libraries and Setup

In [ ]:
import base64
from pathlib import Path

from IPython.display import Markdown, display
from gates_openai import create_response

Read the insurance policy PDF (`2026AnthemgHIPSBC.pdf`) and encode it in base64 so it can be passed to the model as context.

In [ ]:
with Path("../data/2026AnthemgHIPSBC.pdf").open("rb") as file:
    pdf_data = base64.b64encode(file.read()).decode("utf-8")

In [ ]:
pdf_data

## Query the Model

Now you will send a specific query to the model: "How much would I pay for mental health therapy?". You provide the model with a system instruction to act as an expert insurance agent and pass the PDF document alongside the user's text prompt.

In [ ]:
prompt = "How much would I pay for mental health therapy?"

In [ ]:
response = create_response(
    model = "gpt-4o-mini",
    input = [
        {
            "role": "system",
            "content": """You are an expert insurance agent designed to assist with
            coverage queries. Use the provided documents to answer questions
            about insurance policies. If the information is not available in
            the documents, respond accordingly.
            """
        },
        {
            "role":"user",
            "content": [
                {
                    "type": "input_file",
                    "filename": "2026AnthemgHIPSBC.pdf",
                    "file_data": f"data:application/pdf;base64,{pdf_data}",
                },
                {
                    "type": "input_text",
                    "text": prompt
                }
            ]
        }
    ]
)

In [ ]:
display(Markdown(response.output_text))

## Refactor into an Agent Class

To make this code reusable and easier to integrate into an A2A server later, you will wrap the logic into a `PolicyAgent` class in a file named `agents.py`. This class initializes the client and data in the `__init__` method and exposes an `answer_query` method.

In [ ]:
%%writefile agents.py
import base64
from pathlib import Path

from gates_openai import create_response

class PolicyAgent:
    def __init__(self) -> None:
        with Path("../data/2026AnthemgHIPSBC.pdf").open("rb") as file:
            self.pdf_data = base64.standard_b64encode(file.read()).decode("utf-8")

    def answer_query(self, prompt: str) -> str:
        response = create_response(
            model = "gpt-4o-mini",
            input = [
                {
                    "role": "system",
                    "content": """You are an expert insurance agent designed to assist with
                    coverage queries. Use the provided documents to answer questions
                    about insurance policies. If the information is not available in
                    the documents, respond accordingly.
                    """
                },
                {
                    "role":"user",
                    "content": [
                        {
                            "type": "input_file",
                            "filename": "2026AnthemgHIPSBC.pdf",
                            "file_data": f"data:application/pdf;base64,{self.pdf_data}",
                        },
                        {
                            "type": "input_text",
                            "text": prompt
                        }
                    ]
                }
            ]
        )
        
        return response.output_text

## Test the Agent Class

Finally, import the `PolicyAgent` class you just created and test it with the same query to ensure it works as expected.

In [ ]:
from agents import PolicyAgent

print("Running Health Insurance Policy Agent")
agent = PolicyAgent()
prompt = "How much do I pay if I have a hospital stay?"

response = agent.answer_query(prompt)
display(Markdown(response))